# FastAPI Basics for AI Services

This notebook introduces FastAPI fundamentals for building AI backend services. Since FastAPI runs as a web server (not in notebooks), we'll provide code blocks that you can copy to `.py` files and run from the terminal.

## Learning Objectives
- Understand FastAPI project structure
- Create basic endpoints (GET, POST)
- Use Pydantic models for request/response
- Run and test the FastAPI server

## 1. FastAPI Setup and Structure

### Installation

First, install FastAPI and uvicorn (ASGI server):

```bash
pip install fastapi uvicorn pydantic
```

### Project Structure

For a simple AI service:
```
ai-api/
├── main.py          # Main application
├── models.py        # Pydantic models
├── services/        # Business logic
│   └── gemini.py
└── requirements.txt
```

### Minimal FastAPI App

Create `main.py` with this content:

```python
from fastapi import FastAPI

app = FastAPI(
    title="AI Service API",
    description="Backend API for AI-powered services",
    version="1.0.0"
)

@app.get("/")
async def root():
    return {"message": "AI Service API is running"}

@app.get("/health")
async def health_check():
    return {"status": "healthy", "version": "1.0.0"}

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
```

### Running the Server

```bash
# Option 1: Direct execution
python main.py

# Option 2: Using uvicorn with auto-reload
uvicorn main:app --reload --host 0.0.0.0 --port 8000
```

The server will start at `http://localhost:8000`

## 2. Creating Endpoints

### GET Endpoints

GET requests retrieve data without modifying server state:

```python
from fastapi import FastAPI, Query
from typing import Optional

app = FastAPI()

# Simple GET endpoint
@app.get("/models")
async def list_models():
    return {
        "models": [
            {"id": "gemini-pro", "name": "Gemini Pro"},
            {"id": "gemini-flash", "name": "Gemini Flash"}
        ]
    }

# GET with query parameters
@app.get("/models/{model_id}")
async def get_model(
    model_id: str,
    include_info: bool = Query(default=True, description="Include detailed info")
):
    model_info = {
        "id": model_id,
        "name": f"Model {model_id}",
        "max_tokens": 8192
    }
    if not include_info:
        return {"id": model_id}
    return model_info
```

### POST Endpoints

POST requests create or process data. For AI services, this is where inference happens:

```python
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional
import time

app = FastAPI()

# Request model
class ChatRequest(BaseModel):
    message: str = Field(..., min_length=1, max_length=4096, description="User message")
    model: str = Field(default="gemini-pro", description="Model to use")
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)

# Response model
class ChatResponse(BaseModel):
    response: str
    model: str
    latency_ms: float

@app.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest):
    start_time = time.time()
    
    # Simulate AI processing (replace with actual Gemini call)
    ai_response = f"Echo: {request.message}"
    
    latency = (time.time() - start_time) * 1000
    
    return ChatResponse(
        response=ai_response,
        model=request.model,
        latency_ms=latency
    )
```

## 3. Request/Response Models with Pydantic

### Basic Models

Pydantic validates data and provides automatic documentation:

```python
from pydantic import BaseModel, Field, validator
from typing import Optional, List
from enum import Enum

class SentimentType(str, Enum):
    POSITIVE = "positive"
    NEGATIVE = "negative"
    NEUTRAL = "neutral"

class TextClassificationRequest(BaseModel):
    text: str = Field(..., min_length=1, max_length=10000)
    language: str = Field(default="en", pattern=r"^[a-z]{2}$")
    return_confidence: bool = Field(default=True)

class ClassificationResult(BaseModel):
    label: SentimentType
    confidence: float = Field(ge=0.0, le=1.0)
    text_length: int

class TextClassificationResponse(BaseModel):
    results: List[ClassificationResult]
    model: str
    processing_time_ms: float

# Example usage in endpoint
@app.post("/classify", response_model=TextClassificationResponse)
async def classify_text(request: TextClassificationRequest):
    # Classification logic here
    results = [
        ClassificationResult(
            label=SentimentType.POSITIVE,
            confidence=0.95,
            text_length=len(request.text)
        )
    ]
    return TextClassificationResponse(
        results=results,
        model="sentiment-v1",
        processing_time_ms=45.2
    )
```

### Nested Models

Complex request/response structures:

```python
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any

class Message(BaseModel):
    role: str = Field(..., pattern=r"^(user|assistant|system)$")
    content: str = Field(..., min_length=1)

class GenerationConfig(BaseModel):
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    top_p: float = Field(default=0.9, ge=0.0, le=1.0)
    max_output_tokens: Optional[int] = Field(default=None, ge=1)
    stop_sequences: List[str] = Field(default_factory=list)

class ChatCompletionRequest(BaseModel):
    messages: List[Message] = Field(..., min_length=1)
    model: str = Field(default="gemini-pro")
    config: GenerationConfig = Field(default_factory=GenerationConfig)
    stream: bool = Field(default=False)

class UsageInfo(BaseModel):
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int

class ChatCompletionResponse(BaseModel):
    id: str
    choices: List[Dict[str, Any]]
    usage: UsageInfo
    model: str

@app.post("/v1/chat/completions", response_model=ChatCompletionResponse)
async def chat_completions(request: ChatCompletionRequest):
    # OpenAI-compatible response format
    return ChatCompletionResponse(
        id="chatcmpl-123",
        choices=[{"message": {"role": "assistant", "content": "Hello!"}}],
        usage=UsageInfo(prompt_tokens=10, completion_tokens=5, total_tokens=15),
        model=request.model
    )
```

## 4. Error Handling

### Custom Exceptions

```python
from fastapi import FastAPI, HTTPException, status
from fastapi.responses import JSONResponse
from pydantic import BaseModel

app = FastAPI()

class ErrorResponse(BaseModel):
    error: str
    detail: str
    status_code: int

@app.exception_handler(HTTPException)
async def http_exception_handler(request, exc):
    return JSONResponse(
        status_code=exc.status_code,
        content={
            "error": exc.detail,
            "status_code": exc.status_code
        }
    )

@app.post("/predict")
async def predict(data: dict):
    if "text" not in data:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="Missing required field: text"
        )
    
    if len(data["text"]) > 10000:
        raise HTTPException(
            status_code=status.HTTP_422_UNPROCESSABLE_ENTITY,
            detail="Text too long (max 10000 characters)"
        )
    
    return {"result": f"Processed: {data['text'][:50]}..."}
```

## 5. Testing the API

### Using Swagger UI

FastAPI automatically generates interactive documentation:
1. Start the server: `uvicorn main:app --reload`
2. Open browser: `http://localhost:8000/docs`
3. Test endpoints interactively

### Using curl

```bash
# GET request
curl http://localhost:8000/models

# POST request with JSON body
curl -X POST "http://localhost:8000/chat" \
  -H "Content-Type: application/json" \
  -d '{"message": "Hello, AI!"}'

# Classification request
curl -X POST "http://localhost:8000/classify" \
  -H "Content-Type: application/json" \
  -d '{"text": "This movie is amazing!", "language": "en"}'
```

### Using Python requests

```python
import requests

# Test chat endpoint
response = requests.post(
    "http://localhost:8000/chat",
    json={"message": "Hello!"},
    timeout=30
)
print(response.json())

# Test classification endpoint
response = requests.post(
    "http://localhost:8000/classify",
    json={"text": "Great product!", "language": "en"}
)
print(response.json())
```

## Complete Example: Simple AI Chat API

Save this as `main.py` and run with `uvicorn main:app --reload`:

```python
from fastapi import FastAPI, HTTPException, status
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import List, Optional
import time
import uuid

app = FastAPI(
    title="AI Chat API",
    description="Simple AI chat service with FastAPI",
    version="1.0.0"
)

# CORS middleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class ChatMessage(BaseModel):
    role: str = Field(..., pattern=r"^(user|assistant)$")
    content: str = Field(..., min_length=1, max_length=4096)

class ChatRequest(BaseModel):
    message: str = Field(..., min_length=1, max_length=4096, description="User message")
    history: List[ChatMessage] = Field(default_factory=list, description="Conversation history")
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)

class ChatResponse(BaseModel):
    id: str
    response: str
    model: str
    tokens_used: Optional[int]
    latency_ms: float

@app.get("/")
async def root():
    return {"message": "AI Chat API", "version": "1.0.0"}

@app.get("/health")
async def health():
    return {"status": "healthy"}

@app.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest):
    start_time = time.time()
    
    # Simulate AI response (replace with actual Gemini call)
    response_text = f"Echo: {request.message}"
    
    latency = (time.time() - start_time) * 1000
    
    return ChatResponse(
        id=str(uuid.uuid4()),
        response=response_text,
        model="gemini-pro",
        tokens_used=len(response_text.split()),
        latency_ms=latency
    )

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
```

## Key Takeaways

1. **FastAPI** is ideal for AI services due to async support and auto-documentation
2. **Pydantic models** ensure type safety and validation at API boundaries
3. **GET endpoints** for retrieving data, **POST endpoints** for AI inference
4. **Error handling** with HTTPException provides clear error responses
5. **Swagger UI** at `/docs` enables interactive API testing
6. **Always run FastAPI as a server** (not in notebooks) using `uvicorn`

## Next Steps
- Proceed to `02-pydantic-validation.ipynb` for advanced validation patterns
- See `03-ai-api-service.ipynb` for Gemini integration and streaming
- Check `exercises/exercise-01.md` for hands-on practice